In [0]:
from pyspark.sql.functions import sum, count, round

# 1. Четем изчистените данни от Silver слоя
df_silver_read = spark.table("default.silver_transactions")

# 2. Правим бизнес агрегация (Gold слой)
# Цел: Да видим общата сума и броя на транзакциите за всеки акаунт
df_gold = df_silver_read \
    .groupBy("account_id") \
    .agg(
        round(sum("amount"), 2).alias("total_amount_processed"),
        count("transaction_id").alias("transaction_count")
    )

# 3. Визуализираме готовия репорт
display(df_gold)

# 4. Записваме като Managed Table в Gold слоя
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.gold_account_summary")

print("Gold слоят е създаден! Таблица: default.gold_account_summary")